In [ ]:
import pandas as pd
import numpy as np 

In [ ]:
nfl_data = pd.read_csv('testing_dataset/NFL Play by Play 2009-2017 (v4).csv')

np.random.seed(0)

In [ ]:
nfl_data.head()

In [ ]:
# total number of rows
total_rows = nfl_data.shape[0]
print(f'Total number of rows: {total_rows}')

# total nujmber of columns
total_columns = nfl_data.shape[1]
print(f'Total number of columns: {total_columns}')

nfl_data.shape

In [ ]:
missing_values_count = nfl_data.isna().sum()

In [ ]:
missing_values_count.index
# all columns contain atleast one missing value

In [ ]:
missing_values_count.iloc[:11]

In [ ]:
# Calculate the total number of values in the DataFrame,
# total rows multiplied by total columns
total_values = np.prod(nfl_data.shape)
# or  equivalently: total_values = nfl_data.size

# Calculate the total number of missing values in the DataFrame
total_missing = missing_values_count.sum()

print(f"total values: {total_values}\ntotal missing: {total_missing}")
print(f"percent missing: {(total_missing/total_values)*100:.2f}%")


### Figure out why the data is missing

        This is the point at which we get into the part of data science that I like to call "data intution", by which I mean "really looking at your data and trying to figure out why it is the way it is and how that will affect your analysis". It can be a frustrating part of data science, especially if you're newer to the field and don't have a lot of experience.

        For dealing with missing values, you'll need to use your intution to figure out why the value is missing.

        Is this value missing because it wasn't recorded or because it doesn't exist?


        If a value is missing becuase it doesn't exist (like the height of the oldest child of someone who doesn't have any children) then it doesn't make sense to try and guess what it might be. These values you probably do want to keep as NaN. On the other hand, if a value is missing because it wasn't recorded, then you can try to guess what it might have been based on the other values in that column and row. This is called imputation, and we'll learn how to do it next!



Let's work through an example. Looking at the number of missing values in the nfl_data dataframe, I notice that the column "TimesSec" has a lot of missing values in it:

In [ ]:
missing_values_count.iloc[:10]

        Drop missing values
        If you're in a hurry or don't have a reason to figure out why your values are missing, one option you have is to just remove any rows or columns that contain missing values. (Note: I don't generally recommend this approch for important projects! It's usually worth it to take the time to go through your data and really look at all the columns with missing values one-by-one to really get to know your dataset.)

        If you're sure you want to drop rows with missing values, pandas does have a handy function, dropna() to help you do this. Let's try it out on our NFL dataset!

In [ ]:
nfl_data.dropna()

# Drop rows with any missing values, in this case
# it removed all our data because every row has at least one missing value

1️⃣ Functions that reduce or apply

    (apply, sum, mean, min, etc.)

    Here, axis means:

        ➝ “Move along this direction when doing the calculation.”

        axis=0: move down the rows, operate column by column

        axis=1: move across the columns, operate row by row

    df.apply(func, axis=1) “Apply the function to each row.”


2️⃣ Functions that remove labels

    (drop, dropna, rename, etc.)

    Here, axis means:

        ➝ “Which labels are you deleting / modifying?”

        Completely different idea.

        axis=0: remove rows

        axis=1: remove columns



df.dropna(axis=1) “Look at the columns and drop columns that contain NaN.”

In [ ]:
# remove the whole column if any value is NaN
column_na_dropped = nfl_data.dropna(axis=1) 
column_na_dropped.head()

In [ ]:
# how much data was lost

print("Columns in original dataset: %d \n" % nfl_data.shape[1])
print("Columns with na's dropped: %d" % column_na_dropped.shape[1])

# We've lost quite a bit of data, but at this point we 
# have successfully removed all the NaN's from our data.

### Filling in missing values automatically


In [ ]:
subset_nfl_data = nfl_data.loc[:, 'EPA': 'Season'].head()
subset_nfl_data

        We can use the Panda's fillna() function to fill in missing values in a dataframe for us. One option we have is to specify what we want the NaN values to be replaced with. Here, I'm saying that I would like to replace all the NaN values with 0.

In [ ]:
subset_nfl_data.fillna(0)

        I could also be a bit more savvy and replace missing values with whatever value comes directly after it in the same column. (This makes a lot of sense for datasets where the observations have some sort of logical order to them.)

In [ ]:
subset_nfl_data

In [ ]:
# replace all NA's the value that comes directly after it in the same column, 
# then replace all the remaining na's with 0

# subset_nfl_data.fillna(method='bfill', axis=0).fillna(0)

# the above function will be depracated, it does the same.

subset_nfl_data.bfill(axis=0).fillna(0)

These two are effectively the same:

        .bfill(axis=0) = “backward fill” along rows, only for missing values (NaN)
        → for each column, each NaN is replaced by the next non-NaN below it.

        .fillna(0) afterward = any NaNs that still remain (e.g., at the very bottom of a column, or in a column that was all NaNs) are set to 0.

So yes:

        bfill is specifically for filling NaN values using the next valid value.

        It does not change non-missing values.

        Chaining .bfill(...).fillna(0) does exactly what your comment says:

        Fill from the value that comes directly after it in the same column.

        Then replace any remaining NaNs with 0.

### Handling Missing Values

In [ ]:
sf_permits = pd.read_csv('testing_dataset/Building_Permits.csv')

In [ ]:
np.random.seed(0) # only for purposes of reproducibility
sf_permits.shape

In [ ]:
sf_permits.head()

# 2) How many missing data points do we have?

What percentage of the values in the dataset are missing?  Your answer should be a number between 0 and 100.  (If 1/4 of the values in the dataset are missing, the answer is 25.)

In [ ]:
missing_data = sf_permits.isna().sum()
# check how many missing values per column, return a series of counts of missing values per column
missing_data[missing_data > 0].head()

In [ ]:
percent_missing = round((missing_data.sum() / sf_permits.size)*100, 2)

In [ ]:
print(percent_missing)

# 3) Figure out why the data is missing

Look at the columns **"Street Number Suffix"** and **"Zipcode"** from the [San Francisco Building Permits dataset]. Both of these contain missing values. 
- Which, if either, are missing because they don't exist? 
- Which, if either, are missing because they weren't recorded?  

Once you have an answer, run the code cell below.

In [ ]:
sf_permits.loc[:,['Street Number Suffix', 'Zipcode']].agg(['nunique'])
# returns a series of unique value counts for the specified columns

# 4) Drop missing values: rows

If you removed all of the rows of `sf_permits` with missing values, how many rows are left?

**Note**: Do not change the value of `sf_permits` when checking this.

In [ ]:
filter_sf = sf_permits.dropna(axis=0)

In [ ]:
filter_sf.shape
# no rows left after dropping all rows with any missing values, this means that
# every row has at least one missing value

# 5) Drop missing values: columns

Now try removing all the columns with empty values.  
- Create a new DataFrame called `sf_permits_with_na_dropped` that has all of the columns with empty values removed.  
- How many columns were removed from the original `sf_permits` DataFrame? Use this number to set the value of the `dropped_columns` variable below.

In [ ]:
sf_permits_with_na_dropped = sf_permits.dropna(axis=1)

In [ ]:
sf_permits_with_na_dropped.shape

In [ ]:
dropped_columns = sf_permits.shape[1] - sf_permits_with_na_dropped.shape[1]

In [ ]:
dropped_columns

# 6) Fill in missing values automatically
Try replacing all the NaN's in the sf_permits data with the one that comes directly after it and then replacing any remaining NaN's with 0. Set the result to a new DataFrame sf_permits_with_na_imputed.

In [ ]:
sf_permits.head()

In [ ]:
sf_permits_with_na_imputed = sf_permits.bfill(axis=0).fillna(0)

In [ ]:
sf_permits_with_na_imputed.head()

## Scaling and Normalization

Scaling vs. Normalization: What's the difference?
    One of the reasons that it's easy to get confused between scaling and normalization is because the terms are sometimes used interchangeably and, to make it even more confusing, they are very similar! In both cases, you're transforming the values of numeric variables so that the transformed data points have specific helpful properties. The difference is that:

    in scaling, you're changing the range of your data, while
    in normalization, you're changing the shape of the distribution of your data.